In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"

print("Project exists:", os.path.exists(PROJECT_DIR))

print("Dataset exists:",
      os.path.exists(PROJECT_DIR + "/data/air_quality_hourly.parquet"))

print("Neural results exist:",
      os.path.exists(
          PROJECT_DIR + "/results/transferability/neural_loso_all_stations.csv"
      ))

print("Matched ExtraTrees result exists:",
      os.path.exists(
          PROJECT_DIR + "/results/transferability/matched_extratrees/matched_extratrees_Aotizhongxin_1h.csv"
      ))

result_path = (
    PROJECT_DIR +
    "/results/transferability/matched_extratrees/matched_extratrees_Aotizhongxin_1h.csv"
)

if os.path.exists(result_path):
    display(pd.read_csv(result_path))

Project exists: True
Dataset exists: True
Neural results exist: True
Matched ExtraTrees result exists: True


,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
0,ExtraTrees,Aotizhongxin,1,10.660539,20.476935,0.951459,522.811048,5260


In [ ]:
import torch
import os

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

print("Project folder:", PROJECT_DIR)
print("Matched results folder:",
      os.listdir(PROJECT_DIR + "/results/transferability/matched_extratrees"))

GPU available: True
GPU name: Tesla T4
Project folder: /content/drive/MyDrive/AeroSense_AI
Matched results folder: ['matched_extratrees_Aotizhongxin_1h.csv']


In [ ]:
import os
import gc
import time
import glob
import joblib
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from numpy.lib.stride_tricks import sliding_window_view

# Project paths
PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"
MATCHED_DIR = PROJECT_DIR + "/results/transferability/matched_extratrees"
MODEL_DIR = PROJECT_DIR + "/models/transferability"

os.makedirs(MATCHED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Experiment settings
features = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM"
]

TARGET = "PM2.5"
LOOKBACK = 24
HORIZONS = [1, 6, 24]

VALIDATION_START = pd.Timestamp("2015-07-25 14:00:00")
TRAIN_END = pd.Timestamp("2015-12-18 16:00:00")
TEST_START = pd.Timestamp("2016-07-24 20:00:00")

# Reload data
DATA_PATH = PROJECT_DIR + "/data/air_quality_hourly.parquet"

df_transfer = pd.read_parquet(DATA_PATH)
df_transfer["timestamp"] = pd.to_datetime(df_transfer["timestamp"])
df_transfer = df_transfer.sort_values(
    ["station", "timestamp"]
).reset_index(drop=True)

ALL_STATIONS = sorted(df_transfer["station"].unique())

# Prepare station data
station_frames = {}

for station in ALL_STATIONS:
    part = df_transfer[
        df_transfer["station"] == station
    ].sort_values("timestamp").copy()

    part[features] = part[features].ffill()
    station_frames[station] = part

# Fit preprocessing only on training-period observations
training_values = pd.concat(
    [
        part.loc[part["timestamp"] < VALIDATION_START, features]
        for part in station_frames.values()
    ],
    ignore_index=True
)

feature_medians = training_values.median()

scaler = StandardScaler()
scaler.fit(training_values.fillna(feature_medians))

station_arrays = {}

for station, part in station_frames.items():
    values = part[features].fillna(feature_medians)

    station_arrays[station] = {
        "X": scaler.transform(values).astype(np.float32),
        "y": part[TARGET].to_numpy(dtype=np.float32),
        "timestamps": part["timestamp"].to_numpy()
    }


# Build sequences for one station
def build_matched_sequences(station, horizon):
    data = station_arrays[station]

    X = data["X"]
    y = data["y"]
    timestamps = pd.to_datetime(data["timestamps"])

    windows = sliding_window_view(
        X, window_shape=LOOKBACK, axis=0
    ).transpose(0, 2, 1)

    count = len(X) - LOOKBACK + 1 - horizon

    X_seq = windows[:count].reshape(count, -1)

    target_start = LOOKBACK - 1 + horizon
    y_target = y[target_start:target_start + count]
    target_times = timestamps[target_start:target_start + count]

    train_mask = (
        (target_times < VALIDATION_START) &
        np.isfinite(y_target)
    )

    val_mask = (
        (target_times >= VALIDATION_START) &
        (target_times < TRAIN_END) &
        np.isfinite(y_target)
    )

    test_mask = (
        (target_times >= TEST_START) &
        np.isfinite(y_target)
    )

    return {
        "X_train": X_seq[train_mask],
        "y_train": y_target[train_mask],
        "X_val": X_seq[val_mask],
        "y_val": y_target[val_mask],
        "X_test": X_seq[test_mask],
        "y_test": y_target[test_mask],
        "test_times": target_times[test_mask]
    }


# Pool the other 11 stations for training
def build_pooled_training_data(held_out_station, horizon):
    X_train_parts, y_train_parts = [], []
    X_val_parts, y_val_parts = [], []

    for station in ALL_STATIONS:
        if station == held_out_station:
            continue

        data = build_matched_sequences(station, horizon)

        X_train_parts.append(data["X_train"])
        y_train_parts.append(data["y_train"])

        X_val_parts.append(data["X_val"])
        y_val_parts.append(data["y_val"])

    return (
        np.concatenate(X_train_parts),
        np.concatenate(y_train_parts),
        np.concatenate(X_val_parts),
        np.concatenate(y_val_parts)
    )


print("Dataset:", df_transfer.shape)
print("Stations:", len(ALL_STATIONS))
print("Station arrays:", len(station_arrays))
print("Feature count:", len(features))
print("GPU available:", __import__("torch").cuda.is_available())

# Verify saved pilot
pilot_csv = MATCHED_DIR + "/matched_extratrees_Aotizhongxin_1h.csv"
pilot_model = MODEL_DIR + "/matched_extratrees_Aotizhongxin_1h.joblib"

print("\nAotizhongxin 1h result exists:", os.path.exists(pilot_csv))
print("Aotizhongxin 1h model exists:", os.path.exists(pilot_model))

if os.path.exists(pilot_csv):
    display(pd.read_csv(pilot_csv))

Dataset: (420768, 14)
Stations: 12
Station arrays: 12
Feature count: 11
GPU available: True

Aotizhongxin 1h result exists: True
Aotizhongxin 1h model exists: True


,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
0,ExtraTrees,Aotizhongxin,1,10.660539,20.476935,0.951459,522.811048,5260


In [ ]:
def run_matched_extratrees(held_out_station, horizon):

    result_path = (
        MATCHED_DIR
        + f"/matched_extratrees_{held_out_station}_{horizon}h.csv"
    )

    model_path = (
        MODEL_DIR
        + f"/matched_extratrees_{held_out_station}_{horizon}h.joblib"
    )

    # Skip only when BOTH files exist
    if os.path.exists(result_path) and os.path.exists(model_path):
        print(f"SKIPPING: {held_out_station} | {horizon}h")
        return pd.read_csv(result_path).iloc[0].to_dict()

    print(f"\n{'='*65}")
    print(f"Station: {held_out_station} | Horizon: {horizon}h")
    print(f"{'='*65}")

    # Training data from the other 11 stations
    X_train, y_train, X_val, y_val = build_pooled_training_data(
        held_out_station, horizon
    )

    # Test data from the held-out station
    test_data = build_matched_sequences(
        held_out_station, horizon
    )

    X_test = test_data["X_test"]
    y_test = test_data["y_test"]

    print("Train:", X_train.shape)
    print("Validation:", X_val.shape)
    print("Test:", X_test.shape)

    model = ExtraTreesRegressor(
        n_estimators=64,
        max_features=0.5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    start = time.time()
    model.fit(X_train, y_train)
    training_seconds = time.time() - start

    predictions = model.predict(X_test)

    result = pd.DataFrame([{
        "model": "ExtraTrees",
        "station": held_out_station,
        "horizon_hours": horizon,
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
        "training_seconds": training_seconds,
        "test_samples": len(y_test)
    }])

    # Save model first, then results
    joblib.dump(model, model_path)
    result.to_csv(result_path, index=False)

    print("\nCompleted.")
    print("Training time:", round(training_seconds, 2), "seconds")
    display(result)

    del X_train, y_train, X_val, y_val
    del X_test, y_test, predictions, model, test_data
    gc.collect()

    return result.iloc[0].to_dict()


print("Resumable training function created.")
print("Existing completed result files:",
      len(glob.glob(MATCHED_DIR + "/*.csv")))

Resumable training function created.
Existing completed result files: 1


In [ ]:
completed_results = []

for station in ALL_STATIONS:
    for horizon in HORIZONS:

        result = run_matched_extratrees(station, horizon)
        completed_results.append(result)

        print("\nProgress:",
              len(glob.glob(MATCHED_DIR + "/*.csv")),
              "/ 36 result files")

SKIPPING: Aotizhongxin | 1h

Progress: 1 / 36 result files

Station: Aotizhongxin | Horizon: 6h
Train: (231099, 264)
Validation: (38566, 264)
Test: (5260, 264)

Completed.
Training time: 404.57 seconds


,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
0,ExtraTrees,Aotizhongxin,6,33.472872,56.150914,0.635004,404.567021,5260



Progress: 2 / 36 result files

Station: Aotizhongxin | Horizon: 24h
Train: (230901, 264)
Validation: (38566, 264)
Test: (5260, 264)

Completed.
Training time: 387.92 seconds


,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
0,ExtraTrees,Aotizhongxin,24,60.224628,85.624667,0.151265,387.923321,5260



Progress: 3 / 36 result files

Station: Changping | Horizon: 1h
Train: (231154, 264)
Validation: (38566, 264)
Test: (5260, 264)

Completed.
Training time: 409.99 seconds


,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
0,ExtraTrees,Changping,1,10.169403,19.00847,0.944364,409.987637,5260


... (59 outputs left out)

,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
0,ExtraTrees,Wanshouxigong,1,11.694964,24.867993,0.941444,598.833878,5260



Progress: 34 / 36 result files

Station: Wanshouxigong | Horizon: 6h
Train: (231099, 264)
Validation: (38566, 264)
Test: (5260, 264)

Completed.
Training time: 557.94 seconds


,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
0,ExtraTrees,Wanshouxigong,6,37.182797,64.220453,0.609489,557.939219,5260



Progress: 35 / 36 result files

Station: Wanshouxigong | Horizon: 24h
Train: (230901, 264)
Validation: (38566, 264)
Test: (5260, 264)


In [ ]:
import os
import glob

files = glob.glob(MATCHED_DIR + "/*.csv")

print("Completed CSV files:", len(files), "/ 36")

last_file = (
    MATCHED_DIR +
    "/matched_extratrees_Wanshouxigong_24h.csv"
)

print("Final result saved:", os.path.exists(last_file))

if os.path.exists(last_file):
    import pandas as pd
    display(pd.read_csv(last_file))

NameError: name 'MATCHED_DIR' is not defined

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"
MATCHED_DIR = PROJECT_DIR + "/results/transferability/matched_extratrees"
MODEL_DIR = PROJECT_DIR + "/models/transferability"

print("Project folder exists:", os.path.exists(PROJECT_DIR))
print("Results folder exists:", os.path.exists(MATCHED_DIR))
print("Model folder exists:", os.path.exists(MODEL_DIR))

Mounted at /content/drive
Project folder exists: True
Results folder exists: True
Model folder exists: True


In [ ]:
import os
import glob
import pandas as pd

result_files = glob.glob(MATCHED_DIR + "/*.csv")
model_files = glob.glob(MODEL_DIR + "/matched_extratrees_*.joblib")

print("Completed CSV files:", len(result_files), "/ 36")
print("Saved model files:", len(model_files), "/ 36")

final_file = MATCHED_DIR + "/matched_extratrees_Wanshouxigong_24h.csv"

print("\nFinal experiment saved:", os.path.exists(final_file))

if os.path.exists(final_file):
    display(pd.read_csv(final_file))

Completed CSV files: 33 / 36
Saved model files: 9 / 36

Final experiment saved: False


In [ ]:
ALL_STATIONS = [
    "Aotizhongxin", "Changping", "Dingling", "Dongsi",
    "Guanyuan", "Gucheng", "Huairou", "Nongzhanguan",
    "Shunyi", "Tiantan", "Wanliu", "Wanshouxigong"
]

HORIZONS = [1, 6, 24]

existing = {
    os.path.basename(f)
    for f in glob.glob(MATCHED_DIR + "/*.csv")
}

missing = []

for station in ALL_STATIONS:
    for horizon in HORIZONS:
        filename = f"matched_extratrees_{station}_{horizon}h.csv"
        if filename not in existing:
            missing.append((station, horizon))

print("Missing experiments:", len(missing))

for station, horizon in missing:
    print(station, "-", horizon, "hours")

Missing experiments: 3
Wanshouxigong - 1 hours
Wanshouxigong - 6 hours
Wanshouxigong - 24 hours


In [ ]:
for horizon in [1, 6, 24]:
    model_path = (
        MODEL_DIR +
        f"/matched_extratrees_Wanshouxigong_{horizon}h.joblib"
    )

    print(
        f"Wanshouxigong {horizon}h:",
        "MODEL SAVED" if os.path.exists(model_path)
        else "MODEL NOT FOUND"
    )

Wanshouxigong 1h: MODEL NOT FOUND
Wanshouxigong 6h: MODEL NOT FOUND
Wanshouxigong 24h: MODEL NOT FOUND


In [ ]:
print("Training function available:", "run_matched_extratrees" in globals())
print("Training data available:", "df_transfer" in globals())
print("Station list available:", "ALL_STATIONS" in globals())
print("Horizons available:", "HORIZONS" in globals())

Training function available: False
Training data available: False
Station list available: True
Horizons available: True


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import joblib
import glob
import time

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"
MATCHED_DIR = PROJECT_DIR + "/results/transferability/matched_extratrees"
MODEL_DIR = PROJECT_DIR + "/models/transferability"

DATA_PATH = PROJECT_DIR + "/data/air_quality_hourly.parquet"

df_transfer = pd.read_parquet(DATA_PATH)

df_transfer["timestamp"] = pd.to_datetime(df_transfer["timestamp"])
df_transfer = df_transfer.sort_values(
    ["station", "timestamp"]
).reset_index(drop=True)

ALL_STATIONS = sorted(df_transfer["station"].unique().tolist())
HORIZONS = [1, 6, 24]

FEATURES = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM"
]

LOOKBACK = 24

print("Dataset shape:", df_transfer.shape)
print("Stations:", len(ALL_STATIONS))
print("Features:", len(FEATURES))
print("GPU available:", torch.cuda.is_available())
print("Missing result files:", 36 - len(glob.glob(MATCHED_DIR + "/*.csv")))

Dataset shape: (420768, 14)
Stations: 12
Features: 11
GPU available: False
Missing result files: 3


In [ ]:
LOOKBACK = 24
HORIZONS = [1, 6, 24]

timestamps = np.sort(df_transfer["timestamp"].unique())

VALIDATION_START = pd.Timestamp(
    timestamps[int(len(timestamps) * 0.60)]
)

TRAIN_END = pd.Timestamp(
    timestamps[int(len(timestamps) * 0.70)]
)

TEST_START = pd.Timestamp(
    timestamps[int(len(timestamps) * 0.85)]
)

print("Validation start:", VALIDATION_START)
print("Training end:", TRAIN_END)
print("Test start:", TEST_START)

print("\nStations:")
for station in ALL_STATIONS:
    count = (df_transfer["station"] == station).sum()
    print(station, count)

Validation start: 2015-07-25 14:00:00
Training end: 2015-12-18 16:00:00
Test start: 2016-07-24 20:00:00

Stations:
Aotizhongxin 35064
Changping 35064
Dingling 35064
Dongsi 35064
Guanyuan 35064
Gucheng 35064
Huairou 35064
Nongzhanguan 35064
Shunyi 35064
Tiantan 35064
Wanliu 35064
Wanshouxigong 35064


In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

station_arrays = {}

for station in ALL_STATIONS:
    sdf = (
        df_transfer[df_transfer["station"] == station]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    X = sdf[FEATURES].to_numpy(dtype=np.float32)
    y = sdf["PM2.5"].to_numpy(dtype=np.float32)
    ts = pd.to_datetime(sdf["timestamp"]).to_numpy()

    station_arrays[station] = {
        "X": X,
        "y": y,
        "timestamps": ts
    }

def build_matched_sequences(station, horizon):

    data = station_arrays[station]

    X = data["X"]
    y = data["y"]
    ts = data["timestamps"]

    windows = sliding_window_view(
        X, window_shape=LOOKBACK, axis=0
    ).transpose(0, 2, 1)

    target_indices = np.arange(LOOKBACK - 1 + horizon, len(y))

    X_windows = windows[:len(y) - LOOKBACK + 1]
    X_windows = X_windows[:len(target_indices)]

    X_windows = X_windows.reshape(len(X_windows), -1)

    target_times = ts[target_indices]
    targets = y[target_indices]

    valid = np.isfinite(targets)

    train_mask = valid & (target_times < VALIDATION_START)
    val_mask = valid & (
        (target_times >= VALIDATION_START) &
        (target_times < TRAIN_END)
    )
    test_mask = valid & (target_times >= TEST_START)

    return {
        "X_train": X_windows[train_mask],
        "y_train": targets[train_mask],
        "X_val": X_windows[val_mask],
        "y_val": targets[val_mask],
        "X_test": X_windows[test_mask],
        "y_test": targets[test_mask]
    }

print("Sequence preparation function created.")
print("Example station array:", station_arrays["Wanshouxigong"]["X"].shape)

Sequence preparation function created.
Example station array: (35064, 11)


In [ ]:
def build_pooled_training_data(held_out_station, horizon):

    train_X, train_y = [], []
    val_X, val_y = [], []

    for station in ALL_STATIONS:

        if station == held_out_station:
            continue

        data = build_matched_sequences(station, horizon)

        train_X.append(data["X_train"])
        train_y.append(data["y_train"])

        val_X.append(data["X_val"])
        val_y.append(data["y_val"])

    return (
        np.concatenate(train_X),
        np.concatenate(train_y),
        np.concatenate(val_X),
        np.concatenate(val_y)
    )


# Verify the exact missing station and horizon
X_train, y_train, X_val, y_val = build_pooled_training_data(
    "Wanshouxigong", 1
)

test_data = build_matched_sequences("Wanshouxigong", 1)

print("Training shape:", X_train.shape)
print("Training targets:", y_train.shape)
print("Validation shape:", X_val.shape)
print("Validation targets:", y_val.shape)
print("Test shape:", test_data["X_test"].shape)
print("Test targets:", test_data["y_test"].shape)

Training shape: (226428, 264)
Training targets: (226428,)
Validation shape: (37582, 264)
Validation targets: (37582,)
Test shape: (5176, 264)
Test targets: (5176,)


In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def run_matched_extratrees(held_out_station, horizon):

    result_path = (
        MATCHED_DIR +
        f"/matched_extratrees_{held_out_station}_{horizon}h.csv"
    )

    model_path = (
        MODEL_DIR +
        f"/matched_extratrees_{held_out_station}_{horizon}h.joblib"
    )

    os.makedirs(MATCHED_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)

    if os.path.exists(result_path):
        print(f"SKIPPING: {held_out_station} | {horizon}h")
        return pd.read_csv(result_path).iloc[0].to_dict()

    X_train, y_train, X_val, y_val = build_pooled_training_data(
        held_out_station, horizon
    )

    test_data = build_matched_sequences(held_out_station, horizon)

    X_test = test_data["X_test"]
    y_test = test_data["y_test"]

    model = ExtraTreesRegressor(
        n_estimators=64,
        max_features=0.5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_seconds = time.time() - start_time

    predictions = model.predict(X_test)

    result = pd.DataFrame([{
        "model": "ExtraTrees",
        "horizon_hours": horizon,
        "station": held_out_station,
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
        "training_seconds": training_seconds,
        "test_samples": len(y_test)
    }])

    result.to_csv(result_path, index=False)
    joblib.dump(model, model_path)

    print(result.to_string(index=False))
    print("Saved:", result_path)

    return result.iloc[0].to_dict()


print("ExtraTrees recovery function created.")

ExtraTrees recovery function created.


In [ ]:
recovery_results = []

for horizon in [1, 6, 24]:

    print("\n" + "=" * 65)
    print(f"RECOVERING Wanshouxigong — {horizon}h")
    print("=" * 65)

    result = run_matched_extratrees(
        "Wanshouxigong",
        horizon
    )

    recovery_results.append(result)

print("\nALL THREE RECOVERY RUNS FINISHED.")


RECOVERING Wanshouxigong — 1h
     model  horizon_hours       station       mae      rmse       r2  training_seconds  test_samples
ExtraTrees              1 Wanshouxigong 12.020012 25.482337 0.938968       1262.632666          5176
Saved: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees/matched_extratrees_Wanshouxigong_1h.csv

RECOVERING Wanshouxigong — 6h
     model  horizon_hours       station       mae      rmse       r2  training_seconds  test_samples
ExtraTrees              6 Wanshouxigong 37.550527 64.661184 0.607023        1172.32214          5176
Saved: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees/matched_extratrees_Wanshouxigong_6h.csv

RECOVERING Wanshouxigong — 24h
     model  horizon_hours       station       mae      rmse       r2  training_seconds  test_samples
ExtraTrees             24 Wanshouxigong 64.753566 97.617207 0.104363       1180.100736          5176
Saved: /content/drive/MyDrive/AeroSense_AI/resul

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"

print("Project exists:", os.path.exists(PROJECT_DIR))

if os.path.exists(PROJECT_DIR):
    print("\nProject folders:")
    print(os.listdir(PROJECT_DIR))

    print("\nTransferability folder:")
    path = PROJECT_DIR + "/results/transferability"
    print(os.listdir(path) if os.path.exists(path) else "NOT FOUND")

Mounted at /content/drive
Drive mounted: True
Project exists: True

Project folders:
['data', 'notebooks', 'results', 'models', 'reports']

Transferability folder:
['extratrees_leave_one_station_out.csv', 'transferability_summary.csv', 'station_mae_1h.png', 'station_mae_6h.png', 'station_mae_24h.png', 'neural_pilot_Aotizhongxin.csv', 'neural_pilot_Changping.csv', 'neural_pilot_Dingling.csv', 'neural_pilot_Dongsi.csv', 'neural_pilot_Guanyuan.csv', 'neural_pilot_Gucheng.csv', 'neural_pilot_Huairou.csv', 'neural_pilot_Nongzhanguan.csv', 'neural_pilot_Shunyi.csv', 'neural_pilot_Tiantan.csv', 'neural_pilot_Wanliu.csv', 'neural_pilot_Wanshouxigong.csv', 'neural_loso_all_stations.csv', 'neural_loso_summary.csv', 'matched_extratrees']


In [6]:
import os

MATCHED_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "results/transferability/matched_extratrees"
)

print("Folder exists:", os.path.exists(MATCHED_DIR))

if os.path.exists(MATCHED_DIR):
    files = os.listdir(MATCHED_DIR)

    print("Total files:", len(files))

    for f in sorted(files):
        print(f)

Folder exists: True
Total files: 36
matched_extratrees_Aotizhongxin_1h.csv
matched_extratrees_Aotizhongxin_24h.csv
matched_extratrees_Aotizhongxin_6h.csv
matched_extratrees_Changping_1h.csv
matched_extratrees_Changping_24h.csv
matched_extratrees_Changping_6h.csv
matched_extratrees_Dingling_1h.csv
matched_extratrees_Dingling_24h.csv
matched_extratrees_Dingling_6h.csv
matched_extratrees_Dongsi_1h.csv
matched_extratrees_Dongsi_24h.csv
matched_extratrees_Dongsi_6h.csv
matched_extratrees_Guanyuan_1h.csv
matched_extratrees_Guanyuan_24h.csv
matched_extratrees_Guanyuan_6h.csv
matched_extratrees_Gucheng_1h.csv
matched_extratrees_Gucheng_24h.csv
matched_extratrees_Gucheng_6h.csv
matched_extratrees_Huairou_1h.csv
matched_extratrees_Huairou_24h.csv
matched_extratrees_Huairou_6h.csv
matched_extratrees_Nongzhanguan_1h.csv
matched_extratrees_Nongzhanguan_24h.csv
matched_extratrees_Nongzhanguan_6h.csv
matched_extratrees_Shunyi_1h.csv
matched_extratrees_Shunyi_24h.csv
matched_extratrees_Shunyi_6h.csv
m

In [7]:
import os
import glob
import pandas as pd

MATCHED_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "results/transferability/matched_extratrees"
)

files = glob.glob(MATCHED_DIR + "/*.csv")

results = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print("Total CSV files:", len(files))
print("Total result rows:", len(results))
print("Unique stations:", results["station"].nunique())
print("Forecast horizons:", sorted(results["horizon_hours"].unique()))

print("\nMissing station-horizon combinations:")
expected = {
    (s, h)
    for s in sorted(results["station"].unique())
    for h in [1, 6, 24]
}
actual = set(zip(results["station"], results["horizon_hours"]))
print(expected - actual)

print("\nWanshouxigong results:")
display(
    results[results["station"] == "Wanshouxigong"]
    .sort_values("horizon_hours")
)

Total CSV files: 36
Total result rows: 36
Unique stations: 12
Forecast horizons: [np.int64(1), np.int64(6), np.int64(24)]

Missing station-horizon combinations:
set()

Wanshouxigong results:


,model,station,horizon_hours,mae,rmse,r2,training_seconds,test_samples
33,ExtraTrees,Wanshouxigong,1,12.020012,25.482337,0.938968,1262.632666,5176
34,ExtraTrees,Wanshouxigong,6,37.550527,64.661184,0.607023,1172.322140,5176
35,ExtraTrees,Wanshouxigong,24,64.753566,97.617207,0.104363,1180.100736,5176


In [8]:
import pandas as pd
import glob
import os

MATCHED_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "results/transferability/matched_extratrees"
)

files = glob.glob(MATCHED_DIR + "/*.csv")

results = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)

print("TEST SAMPLE COUNTS BY HORIZON")
display(
    results.groupby("horizon_hours")["test_samples"]
    .agg(["count", "min", "max", "mean", "std"])
)

print("\nEXPERIMENTS WITH UNEXPECTED SAMPLE COUNTS")

for horizon in [1, 6, 24]:
    subset = results[results["horizon_hours"] == horizon]

    print(f"\n{horizon}h:")
    print(
        subset[["station", "test_samples"]]
        .sort_values("test_samples")
        .to_string(index=False)
    )


TEST SAMPLE COUNTS BY HORIZON


,count,min,max,mean,std
horizon_hours,,,,,
1,12,5176,5260,5253.0,24.248711
6,12,5176,5260,5253.0,24.248711
24,12,5176,5260,5253.0,24.248711



EXPERIMENTS WITH UNEXPECTED SAMPLE COUNTS

1h:
      station  test_samples
Wanshouxigong          5176
 Aotizhongxin          5260
     Dingling          5260
    Changping          5260
     Guanyuan          5260
      Gucheng          5260
      Huairou          5260
       Dongsi          5260
 Nongzhanguan          5260
       Shunyi          5260
      Tiantan          5260
       Wanliu          5260

6h:
      station  test_samples
Wanshouxigong          5176
 Aotizhongxin          5260
     Dingling          5260
    Changping          5260
     Guanyuan          5260
      Gucheng          5260
      Huairou          5260
       Dongsi          5260
 Nongzhanguan          5260
       Shunyi          5260
      Tiantan          5260
       Wanliu          5260

24h:
      station  test_samples
Wanshouxigong          5176
 Aotizhongxin          5260
     Dingling          5260
    Changping          5260
     Guanyuan          5260
      Gucheng          5260
      Huairou    

In [9]:
import pandas as pd
import numpy as np

df_audit = pd.read_parquet(
    "/content/drive/MyDrive/AeroSense_AI/data/air_quality_hourly.parquet"
)

df_audit["timestamp"] = pd.to_datetime(df_audit["timestamp"])

TEST_START = pd.Timestamp("2016-07-24 20:00:00")

print("TEST PERIOD TARGET MISSINGNESS\n")

for station in sorted(df_audit["station"].unique()):

    sdf = df_audit[
        (df_audit["station"] == station) &
        (df_audit["timestamp"] >= TEST_START)
    ]

    total = len(sdf)
    missing = sdf["PM2.5"].isna().sum()
    valid = sdf["PM2.5"].notna().sum()

    print(
        f"{station:18s} | "
        f"Total: {total} | "
        f"Valid: {valid} | "
        f"Missing PM2.5: {missing}"
    )

TEST PERIOD TARGET MISSINGNESS

Aotizhongxin       | Total: 5260 | Valid: 5173 | Missing PM2.5: 87
Changping          | Total: 5260 | Valid: 5201 | Missing PM2.5: 59
Dingling           | Total: 5260 | Valid: 5045 | Missing PM2.5: 215
Dongsi             | Total: 5260 | Valid: 5082 | Missing PM2.5: 178
Guanyuan           | Total: 5260 | Valid: 5159 | Missing PM2.5: 101
Gucheng            | Total: 5260 | Valid: 5160 | Missing PM2.5: 100
Huairou            | Total: 5260 | Valid: 5176 | Missing PM2.5: 84
Nongzhanguan       | Total: 5260 | Valid: 5172 | Missing PM2.5: 88
Shunyi             | Total: 5260 | Valid: 5113 | Missing PM2.5: 147
Tiantan            | Total: 5260 | Valid: 5185 | Missing PM2.5: 75
Wanliu             | Total: 5260 | Valid: 5195 | Missing PM2.5: 65
Wanshouxigong      | Total: 5260 | Valid: 5176 | Missing PM2.5: 84


In [11]:
import os
import glob
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"

DATA_PATH = PROJECT_DIR + "/data/air_quality_hourly.parquet"

df_transfer = pd.read_parquet(DATA_PATH)
df_transfer["timestamp"] = pd.to_datetime(df_transfer["timestamp"])

ALL_STATIONS = sorted(df_transfer["station"].unique().tolist())
HORIZONS = [1, 6, 24]

FEATURES = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM"
]

LOOKBACK = 24

timestamps = np.sort(df_transfer["timestamp"].unique())

VALIDATION_START = pd.Timestamp(timestamps[int(len(timestamps) * 0.60)])
TRAIN_END = pd.Timestamp(timestamps[int(len(timestamps) * 0.70)])
TEST_START = pd.Timestamp(timestamps[int(len(timestamps) * 0.85)])

print("Environment restored.")
print("Stations:", len(ALL_STATIONS))
print("Horizons:", HORIZONS)
print("Dataset:", df_transfer.shape)
print("Test start:", TEST_START)

Environment restored.
Stations: 12
Horizons: [1, 6, 24]
Dataset: (420768, 14)
Test start: 2016-07-24 20:00:00


In [12]:
from numpy.lib.stride_tricks import sliding_window_view

station_arrays = {}

for station in ALL_STATIONS:
    sdf = (
        df_transfer[df_transfer["station"] == station]
        .sort_values("timestamp")
        .reset_index(drop=True)
    )

    station_arrays[station] = {
        "X": sdf[FEATURES].to_numpy(dtype=np.float32),
        "y": sdf["PM2.5"].to_numpy(dtype=np.float32),
        "timestamps": sdf["timestamp"].to_numpy()
    }


def build_matched_sequences(station, horizon):

    data = station_arrays[station]

    X = data["X"]
    y = data["y"]
    ts = data["timestamps"]

    windows = sliding_window_view(
        X, window_shape=LOOKBACK, axis=0
    ).transpose(0, 2, 1)

    target_indices = np.arange(
        LOOKBACK - 1 + horizon, len(y)
    )

    X_windows = windows[:len(target_indices)]
    X_windows = X_windows.reshape(len(X_windows), -1)

    targets = y[target_indices]
    target_times = ts[target_indices]

    valid = np.isfinite(targets)

    train_mask = valid & (target_times < VALIDATION_START)

    val_mask = valid & (
        (target_times >= VALIDATION_START) &
        (target_times < TRAIN_END)
    )

    test_mask = valid & (target_times >= TEST_START)

    return {
        "X_train": X_windows[train_mask],
        "y_train": targets[train_mask],
        "X_val": X_windows[val_mask],
        "y_val": targets[val_mask],
        "X_test": X_windows[test_mask],
        "y_test": targets[test_mask]
    }


sample_audit = []

for station in ALL_STATIONS:
    for horizon in HORIZONS:

        data = build_matched_sequences(station, horizon)

        sample_audit.append({
            "station": station,
            "horizon": horizon,
            "train_samples": len(data["X_train"]),
            "validation_samples": len(data["X_val"]),
            "test_samples": len(data["X_test"])
        })

sample_audit_df = pd.DataFrame(sample_audit)

print("Total test samples:", sample_audit_df["test_samples"].sum())

display(
    sample_audit_df.pivot(
        index="station",
        columns="horizon",
        values="test_samples"
    )
)

Total test samples: 185511


horizon,1,6,24
station,,,
Aotizhongxin,5173,5173,5173
Changping,5201,5201,5201
Dingling,5045,5045,5045
Dongsi,5082,5082,5082
Guanyuan,5159,5159,5159
Gucheng,5160,5160,5160
Huairou,5176,5176,5176
Nongzhanguan,5172,5172,5172
Shunyi,5113,5113,5113


In [13]:
import os
import shutil
import glob

BASE_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "results/transferability"
)

OLD_DIR = BASE_DIR + "/matched_extratrees"
CORRECTED_DIR = BASE_DIR + "/matched_extratrees_corrected"

os.makedirs(CORRECTED_DIR, exist_ok=True)

# Preserve the three corrected Wanshouxigong results
for horizon in [1, 6, 24]:

    filename = f"matched_extratrees_Wanshouxigong_{horizon}h.csv"

    source = os.path.join(OLD_DIR, filename)
    destination = os.path.join(CORRECTED_DIR, filename)

    shutil.copy2(source, destination)

print("Corrected results folder created.")
print("Existing corrected results:", len(glob.glob(CORRECTED_DIR + "/*.csv")))
print("Remaining experiments:", 33)

Corrected results folder created.
Existing corrected results: 3
Remaining experiments: 33


In [14]:
def build_pooled_training_data(held_out_station, horizon):

    train_X, train_y = [], []
    val_X, val_y = [], []

    for station in ALL_STATIONS:

        if station == held_out_station:
            continue

        data = build_matched_sequences(station, horizon)

        train_X.append(data["X_train"])
        train_y.append(data["y_train"])

        val_X.append(data["X_val"])
        val_y.append(data["y_val"])

    return (
        np.concatenate(train_X),
        np.concatenate(train_y),
        np.concatenate(val_X),
        np.concatenate(val_y)
    )


X_train, y_train, X_val, y_val = build_pooled_training_data(
    "Aotizhongxin", 1
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Training targets:", y_train.shape)
print("Validation targets:", y_val.shape)

Training shape: (226706, 264)
Validation shape: (37585, 264)
Training targets: (226706,)
Validation targets: (37585,)


In [15]:
import os
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CORRECTED_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "results/transferability/matched_extratrees_corrected"
)

MODEL_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "models/transferability/matched_extratrees_corrected"
)

os.makedirs(CORRECTED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


def run_corrected_extratrees(held_out_station, horizon):

    filename = f"matched_extratrees_{held_out_station}_{horizon}h.csv"
    result_path = os.path.join(CORRECTED_DIR, filename)

    model_path = os.path.join(
        MODEL_DIR,
        f"matched_extratrees_{held_out_station}_{horizon}h.joblib"
    )

    if os.path.exists(result_path):
        print("Already completed:", filename)
        return pd.read_csv(result_path).iloc[0].to_dict()

    X_train, y_train, X_val, y_val = build_pooled_training_data(
        held_out_station, horizon
    )

    test_data = build_matched_sequences(held_out_station, horizon)

    X_test = test_data["X_test"]
    y_test = test_data["y_test"]

    model = ExtraTreesRegressor(
        n_estimators=64,
        max_features=0.5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    print(f"\nTraining: {held_out_station} | {horizon}h")
    print("Train:", len(y_train))
    print("Validation:", len(y_val))
    print("Test:", len(y_test))

    start = time.time()
    model.fit(X_train, y_train)
    training_seconds = time.time() - start

    predictions = model.predict(X_test)

    result = pd.DataFrame([{
        "model": "ExtraTrees",
        "horizon_hours": horizon,
        "station": held_out_station,
        "train_samples": len(y_train),
        "validation_samples": len(y_val),
        "test_samples": len(y_test),
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
        "training_seconds": training_seconds
    }])

    result.to_csv(result_path, index=False)

    import joblib
    joblib.dump(model, model_path)

    print("\nRESULT:")
    print(result.to_string(index=False))
    print("Saved:", result_path)

    return result.iloc[0].to_dict()

In [16]:
run_corrected_extratrees("Aotizhongxin", 1)


Training: Aotizhongxin | 1h
Train: 226706
Validation: 37585
Test: 5173

RESULT:
     model  horizon_hours      station  train_samples  validation_samples  test_samples       mae      rmse       r2  training_seconds
ExtraTrees              1 Aotizhongxin         226706               37585          5173 10.880356 20.464523 0.952015       1095.064383
Saved: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_corrected/matched_extratrees_Aotizhongxin_1h.csv


{'model': 'ExtraTrees',
 'horizon_hours': 1,
 'station': 'Aotizhongxin',
 'train_samples': 226706,
 'validation_samples': 37585,
 'test_samples': 5173,
 'mae': 10.880355531860959,
 'rmse': 20.46452323365108,
 'r2': 0.9520154236214609,
 'training_seconds': 1095.0643830299377}

In [2]:
import os
import time
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CORRECTED_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "results/transferability/matched_extratrees_corrected"
)

MODEL_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/"
    "models/transferability/matched_extratrees_corrected"
)

os.makedirs(CORRECTED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)


def run_corrected_extratrees(held_out_station, horizon):

    filename = f"matched_extratrees_{held_out_station}_{horizon}h.csv"
    result_path = os.path.join(CORRECTED_DIR, filename)

    model_path = os.path.join(
        MODEL_DIR,
        f"matched_extratrees_{held_out_station}_{horizon}h.joblib"
    )

    if os.path.exists(result_path):
        print("Already completed:", filename)
        return pd.read_csv(result_path).iloc[0].to_dict()

    X_train, y_train, X_val, y_val = build_pooled_training_data(
        held_out_station, horizon
    )

    test_data = build_matched_sequences(held_out_station, horizon)

    X_test = test_data["X_test"]
    y_test = test_data["y_test"]

    model = ExtraTreesRegressor(
        n_estimators=64,
        max_features=0.5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    print(f"\nTraining: {held_out_station} | {horizon}h")
    print("Train:", len(y_train))
    print("Validation:", len(y_val))
    print("Test:", len(y_test))

    start = time.time()
    model.fit(X_train, y_train)
    training_seconds = time.time() - start

    predictions = model.predict(X_test)

    result = pd.DataFrame([{
        "model": "ExtraTrees",
        "horizon_hours": horizon,
        "station": held_out_station,
        "train_samples": len(y_train),
        "validation_samples": len(y_val),
        "test_samples": len(y_test),
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": np.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
        "training_seconds": training_seconds
    }])

    result.to_csv(result_path, index=False)
    joblib.dump(model, model_path)

    print("\nRESULT:")
    print(result.to_string(index=False))
    print("Saved:", result_path)

    return result.iloc[0].to_dict()

print("Training function restored.")

Training function restored.


In [3]:
print("Pooled function:", callable(globals().get("build_pooled_training_data")))
print("Sequence function:", callable(globals().get("build_matched_sequences")))
print("Training function:", callable(globals().get("run_corrected_extratrees")))

Pooled function: False
Sequence function: False
Training function: True


In [2]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

print("MyDrive exists:", os.path.exists("/content/drive/MyDrive"))

print("\nFolders inside MyDrive:")
print(os.listdir("/content/drive/MyDrive")[:30])

Mounted at /content/drive
MyDrive exists: True

Folders inside MyDrive:
['III - B  TermII Internal marks.xlsx', 'Int.MarksTerm2Final.xlsx', 'Contacts-2024-06-16.vcf', 'Uma Contacts-2024-06-16.vcf', 'Certificate.pdf', 'certificate.pdf', 'Student details excel  (24-25) iii B.xlsx', 'Colab Notebooks', 'Wavs.zip', 'colab', 'all_json.zip', 'all_json', 'adhi_output_final.zip', 'janani_output_final.zip', 'veda_output_final.zip', 'madhav_output_final.zip', 'nitheesh_output_final.zip', 'sasi_output_final.zip', 'voice', '.pdf (1).pdf', '.pdf.pdf', 'Certificate_copy.pdf', 'Attandance- CBSE Teachers.xlsx', 'AeroSense_AI']


In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"

print("Project folder contents:")
for item in os.listdir(PROJECT_DIR):
    path = os.path.join(PROJECT_DIR, item)
    if os.path.isdir(path):
        print(f"[FOLDER] {item}")
    else:
        print(f"[FILE]   {item}")


Project folder contents:
[FOLDER] data
[FOLDER] notebooks
[FOLDER] results
[FOLDER] models
[FOLDER] reports


In [4]:
import os

DATA_DIR = "/content/drive/MyDrive/AeroSense_AI/data"

print("Data folder contents:")
for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  {file}")


Data folder contents:
data/
  air_quality_hourly.parquet


In [5]:
import pandas as pd
import os

DATA_PATH = "/content/drive/MyDrive/AeroSense_AI/data/air_quality_hourly.parquet"

print("File exists:", os.path.exists(DATA_PATH))
print("File size (MB):", round(os.path.getsize(DATA_PATH) / (1024**2), 2))

df_transfer = pd.read_parquet(DATA_PATH)

print("\nDataset loaded successfully!")
print("Shape:", df_transfer.shape)
print("Columns:", df_transfer.columns.tolist())
print("Stations:", df_transfer["station"].nunique())

File exists: True
File size (MB): 5.97

Dataset loaded successfully!
Shape: (420768, 14)
Columns: ['timestamp', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']
Stations: 12


In [6]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"
DATA_PATH = PROJECT_DIR + "/data/air_quality_hourly.parquet"

df_transfer = pd.read_parquet(DATA_PATH)
df_transfer["timestamp"] = pd.to_datetime(df_transfer["timestamp"])
df_transfer = df_transfer.sort_values(
    ["station", "timestamp"]
).reset_index(drop=True)

FEATURES = [
    "PM2.5", "PM10", "SO2", "NO2", "CO", "O3",
    "TEMP", "PRES", "DEWP", "RAIN", "WSPM"
]

TARGET = "PM2.5"
LOOKBACK = 24
HORIZONS = [1, 6, 24]

VALIDATION_START = pd.Timestamp("2015-07-25 14:00:00")
TRAIN_END = pd.Timestamp("2015-12-18 16:00:00")
TEST_START = pd.Timestamp("2016-07-24 20:00:00")

ALL_STATIONS = sorted(df_transfer["station"].unique())

station_frames = {}

for station in ALL_STATIONS:
    part = df_transfer[
        df_transfer["station"] == station
    ].sort_values("timestamp").copy()

    part[FEATURES] = part[FEATURES].ffill()
    station_frames[station] = part

training_values = pd.concat(
    [
        part.loc[
            part["timestamp"] < VALIDATION_START,
            FEATURES
        ]
        for part in station_frames.values()
    ],
    ignore_index=True
)

feature_medians = training_values.median()

scaler = StandardScaler()
scaler.fit(training_values.fillna(feature_medians))

station_arrays = {}

for station, part in station_frames.items():
    values = part[FEATURES].fillna(feature_medians)

    station_arrays[station] = {
        "X": scaler.transform(values).astype(np.float32),
        "y": part[TARGET].to_numpy(dtype=np.float32),
        "timestamps": part["timestamp"].to_numpy()
    }

print("Dataset:", df_transfer.shape)
print("Stations:", len(ALL_STATIONS))
print("Station arrays:", len(station_arrays))
print("Preprocessing restored.")

Dataset: (420768, 14)
Stations: 12
Station arrays: 12
Preprocessing restored.


In [7]:
def build_matched_sequences(station, horizon):

    data = station_arrays[station]

    X = data["X"]
    y = data["y"]
    timestamps = pd.to_datetime(data["timestamps"])

    windows = sliding_window_view(
        X, window_shape=LOOKBACK, axis=0
    ).transpose(0, 2, 1)

    count = len(X) - LOOKBACK + 1 - horizon

    X_seq = windows[:count].reshape(count, -1)

    target_start = LOOKBACK - 1 + horizon
    y_target = y[target_start:target_start + count]
    target_times = timestamps[target_start:target_start + count]

    train_mask = (
        (target_times < VALIDATION_START) &
        np.isfinite(y_target)
    )

    val_mask = (
        (target_times >= VALIDATION_START) &
        (target_times < TRAIN_END) &
        np.isfinite(y_target)
    )

    test_mask = (
        (target_times >= TEST_START) &
        np.isfinite(y_target)
    )

    return {
        "X_train": X_seq[train_mask],
        "y_train": y_target[train_mask],
        "X_val": X_seq[val_mask],
        "y_val": y_target[val_mask],
        "X_test": X_seq[test_mask],
        "y_test": y_target[test_mask],
        "test_times": target_times[test_mask]
    }


def build_pooled_training_data(held_out_station, horizon):

    train_X, train_y = [], []
    val_X, val_y = [], []

    for station in ALL_STATIONS:

        if station == held_out_station:
            continue

        data = build_matched_sequences(station, horizon)

        train_X.append(data["X_train"])
        train_y.append(data["y_train"])

        val_X.append(data["X_val"])
        val_y.append(data["y_val"])

    return (
        np.concatenate(train_X),
        np.concatenate(train_y),
        np.concatenate(val_X),
        np.concatenate(val_y)
    )


print("Sequence function:", callable(build_matched_sequences))
print("Pooled function:", callable(build_pooled_training_data))

Sequence function: True
Pooled function: True


In [8]:
X_train, y_train, X_val, y_val = build_pooled_training_data(
    "Aotizhongxin", 6
)

test_data = build_matched_sequences("Aotizhongxin", 6)

print("Held-out station: Aotizhongxin")
print("Forecast horizon: 6 hours")

print("\nTraining:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", test_data["X_test"].shape, test_data["y_test"].shape)

print("\nMissing training targets:", np.isnan(y_train).sum())
print("Missing validation targets:", np.isnan(y_val).sum())
print("Missing test targets:", np.isnan(test_data["y_test"]).sum())

Held-out station: Aotizhongxin
Forecast horizon: 6 hours

Training: (231099, 264) (231099,)
Validation: (38566, 264) (38566,)
Testing: (5260, 264) (5260,)

Missing training targets: 0
Missing validation targets: 0
Missing test targets: 0


In [11]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AeroSense_AI"

CORRECTED_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "transferability",
    "matched_extratrees_corrected"
)

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "transferability",
    "matched_extratrees_corrected"
)

os.makedirs(CORRECTED_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print("Corrected results folder:", CORRECTED_DIR)
print("Model folder:", MODEL_DIR)

print("\nExisting result files:")
files = sorted(os.listdir(CORRECTED_DIR))

for file in files:
    print(file)

print("\nTotal saved result files:", len(files))

Corrected results folder: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_corrected
Model folder: /content/drive/MyDrive/AeroSense_AI/models/transferability/matched_extratrees_corrected

Existing result files:
matched_extratrees_Aotizhongxin_1h.csv
matched_extratrees_Wanshouxigong_1h.csv
matched_extratrees_Wanshouxigong_24h.csv
matched_extratrees_Wanshouxigong_6h.csv

Total saved result files: 4


In [12]:
FINAL_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "transferability",
    "matched_extratrees_final"
)

FINAL_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "transferability",
    "matched_extratrees_final"
)

os.makedirs(FINAL_DIR, exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

print("Final results folder:", FINAL_DIR)
print("Final model folder:", FINAL_MODEL_DIR)

print("Existing final results:", len(os.listdir(FINAL_DIR)))

Final results folder: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_final
Final model folder: /content/drive/MyDrive/AeroSense_AI/models/transferability/matched_extratrees_final
Existing final results: 0


In [13]:
import time
import joblib
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def run_final_extratrees(held_out_station, horizon):

    result_file = os.path.join(
        FINAL_DIR,
        f"matched_extratrees_{held_out_station}_{horizon}h.csv"
    )

    model_file = os.path.join(
        FINAL_MODEL_DIR,
        f"matched_extratrees_{held_out_station}_{horizon}h.joblib"
    )

    if os.path.exists(result_file):
        print(f"Already completed: {held_out_station}, {horizon}h")
        return pd.read_csv(result_file)

    print("=" * 65)
    print(f"Training ExtraTrees: {held_out_station} | Horizon: {horizon}h")
    print("=" * 65)

    start_time = time.time()

    X_train, y_train, X_val, y_val = build_pooled_training_data(
        held_out_station, horizon
    )

    test_data = build_matched_sequences(
        held_out_station, horizon
    )

    X_test = test_data["X_test"]
    y_test = test_data["y_test"]

    print("Training samples:", len(y_train))
    print("Validation samples:", len(y_val))
    print("Test samples:", len(y_test))

    model = ExtraTreesRegressor(
        n_estimators=64,
        max_features=0.5,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    training_seconds = time.time() - start_time

    result = pd.DataFrame([{
        "model": "ExtraTrees",
        "held_out_station": held_out_station,
        "horizon_hours": horizon,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "training_seconds": training_seconds,
        "test_samples": len(y_test)
    }])

    joblib.dump(model, model_file)
    result.to_csv(result_file, index=False)

    print("\nRESULT")
    print(result.to_string(index=False))
    print(f"\nSaved: {result_file}")
    print(f"Elapsed time: {training_seconds / 60:.2f} minutes")

    return result


print("Training function restored:", callable(run_final_extratrees))
print("Final results folder:", FINAL_DIR)

Training function restored: True
Final results folder: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_final


In [14]:
result = run_final_extratrees("Aotizhongxin", 6)

Training ExtraTrees: Aotizhongxin | Horizon: 6h
Training samples: 231099
Validation samples: 38566
Test samples: 5260

RESULT
     model held_out_station  horizon_hours       mae      rmse       r2  training_seconds  test_samples
ExtraTrees     Aotizhongxin              6 33.472872 56.150914 0.635004         354.64475          5260

Saved: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_final/matched_extratrees_Aotizhongxin_6h.csv
Elapsed time: 5.91 minutes


In [15]:
completed = 0
skipped = 0
failed = []

for station in ALL_STATIONS:
    for horizon in HORIZONS:

        filename = (
            f"matched_extratrees_{station}_{horizon}h.csv"
        )

        result_path = os.path.join(FINAL_DIR, filename)

        if os.path.exists(result_path):
            print(f"SKIPPING: {station} | {horizon}h")
            skipped += 1
            continue

        try:
            run_final_extratrees(station, horizon)
            completed += 1

        except Exception as e:
            print(f"FAILED: {station} | {horizon}h")
            print("Error:", repr(e))
            failed.append((station, horizon, repr(e)))

            # Stop here so we can inspect the error.
            break

    if failed:
        break

print("\n========== BATCH SUMMARY ==========")
print("Completed in this run:", completed)
print("Skipped existing results:", skipped)
print("Failures:", len(failed))
print("Total CSV files:", len([
    f for f in os.listdir(FINAL_DIR)
    if f.endswith(".csv")
]))
print("Failed experiments:", failed)

Training ExtraTrees: Aotizhongxin | Horizon: 1h
Training samples: 231154
Validation samples: 38566
Test samples: 5260

RESULT
     model held_out_station  horizon_hours       mae      rmse       r2  training_seconds  test_samples
ExtraTrees     Aotizhongxin              1 10.660539 20.476935 0.951459        389.516797          5260

Saved: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_final/matched_extratrees_Aotizhongxin_1h.csv
Elapsed time: 6.49 minutes
SKIPPING: Aotizhongxin | 6h
Training ExtraTrees: Aotizhongxin | Horizon: 24h
Training samples: 230901
Validation samples: 38566
Test samples: 5260

RESULT
     model held_out_station  horizon_hours       mae      rmse       r2  training_seconds  test_samples
ExtraTrees     Aotizhongxin             24 60.224628 85.624667 0.151265        370.902574          5260

Saved: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_final/matched_extratrees_Aotizhongxin_24h.csv
Elapsed tim

In [16]:
part[FEATURES] = part[FEATURES].ffill()

In [17]:
# Preserve original PM2.5 values before any imputation
station_frames = {}
station_arrays = {}

for station in ALL_STATIONS:

    part = df_transfer[
        df_transfer["station"] == station
    ].sort_values("timestamp").copy()

    # Preserve genuine observed targets
    raw_target = part[TARGET].to_numpy(dtype=np.float32).copy()

    # Impute input features only
    part[FEATURES] = part[FEATURES].ffill()

    station_frames[station] = {
        "frame": part,
        "raw_target": raw_target
    }

# Calculate medians using training-period feature observations
training_values = pd.concat(
    [
        item["frame"].loc[
            item["frame"]["timestamp"] < VALIDATION_START,
            FEATURES
        ]
        for item in station_frames.values()
    ],
    ignore_index=True
)

feature_medians = training_values.median()

scaler = StandardScaler()
scaler.fit(training_values.fillna(feature_medians))

# Rebuild station arrays with untouched target values
for station, item in station_frames.items():

    part = item["frame"]
    raw_target = item["raw_target"]

    values = part[FEATURES].fillna(feature_medians)

    station_arrays[station] = {
        "X": scaler.transform(values).astype(np.float32),
        "y": raw_target,
        "timestamps": part["timestamp"].to_numpy()
    }

print("Corrected station arrays:", len(station_arrays))

for station in ALL_STATIONS:
    valid = np.isfinite(station_arrays[station]["y"])
    print(
        station,
        "| Valid PM2.5 targets:",
        int(valid.sum()),
        "| Missing targets:",
        int((~valid).sum())
    )

Corrected station arrays: 12
Aotizhongxin | Valid PM2.5 targets: 34139 | Missing targets: 925
Changping | Valid PM2.5 targets: 34290 | Missing targets: 774
Dingling | Valid PM2.5 targets: 34285 | Missing targets: 779
Dongsi | Valid PM2.5 targets: 34314 | Missing targets: 750
Guanyuan | Valid PM2.5 targets: 34448 | Missing targets: 616
Gucheng | Valid PM2.5 targets: 34418 | Missing targets: 646
Huairou | Valid PM2.5 targets: 34111 | Missing targets: 953
Nongzhanguan | Valid PM2.5 targets: 34436 | Missing targets: 628
Shunyi | Valid PM2.5 targets: 34151 | Missing targets: 913
Tiantan | Valid PM2.5 targets: 34387 | Missing targets: 677
Wanliu | Valid PM2.5 targets: 34682 | Missing targets: 382
Wanshouxigong | Valid PM2.5 targets: 34368 | Missing targets: 696


In [18]:
import inspect

print("Sequence builder signature:")
print(inspect.signature(build_matched_sequences))

print("\nSequence builder source:")
print(inspect.getsource(build_matched_sequences))

Sequence builder signature:
(station, horizon)

Sequence builder source:
def build_matched_sequences(station, horizon):

    data = station_arrays[station]

    X = data["X"]
    y = data["y"]
    timestamps = pd.to_datetime(data["timestamps"])

    windows = sliding_window_view(
        X, window_shape=LOOKBACK, axis=0
    ).transpose(0, 2, 1)

    count = len(X) - LOOKBACK + 1 - horizon

    X_seq = windows[:count].reshape(count, -1)

    target_start = LOOKBACK - 1 + horizon
    y_target = y[target_start:target_start + count]
    target_times = timestamps[target_start:target_start + count]

    train_mask = (
        (target_times < VALIDATION_START) &
        np.isfinite(y_target)
    )

    val_mask = (
        (target_times >= VALIDATION_START) &
        (target_times < TRAIN_END) &
        np.isfinite(y_target)
    )

    test_mask = (
        (target_times >= TEST_START) &
        np.isfinite(y_target)
    )

    return {
        "X_train": X_seq[train_mask],
        "y_train":

In [19]:
# Audit corrected sequence counts before training

print("STATION          | HORIZON | TRAIN  | VALIDATION | TEST")
print("-" * 65)

for station in ALL_STATIONS:
    for horizon in [1, 6, 24]:

        seq = build_matched_sequences(station, horizon)

        print(
            f"{station:16s} | "
            f"{horizon:7d} | "
            f"{len(seq['X_train']):6d} | "
            f"{len(seq['X_val']):10d} | "
            f"{len(seq['X_test']):5d}"
        )


STATION          | HORIZON | TRAIN  | VALIDATION | TEST
-----------------------------------------------------------------
Aotizhongxin     |       1 |  20384 |       3406 |  5173
Aotizhongxin     |       6 |  20379 |       3406 |  5173
Aotizhongxin     |      24 |  20361 |       3406 |  5173
Changping        |       1 |  20586 |       3293 |  5201
Changping        |       6 |  20581 |       3293 |  5201
Changping        |      24 |  20563 |       3293 |  5201
Dingling         |       1 |  20619 |       3440 |  5045
Dingling         |       6 |  20614 |       3440 |  5045
Dingling         |      24 |  20596 |       3440 |  5045
Dongsi           |       1 |  20688 |       3467 |  5082
Dongsi           |       6 |  20683 |       3467 |  5082
Dongsi           |      24 |  20665 |       3467 |  5082
Guanyuan         |       1 |  20634 |       3464 |  5159
Guanyuan         |       6 |  20629 |       3464 |  5159
Guanyuan         |      24 |  20611 |       3464 |  5159
Gucheng          |     

In [20]:
from pathlib import Path

FINAL_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/results/"
    "transferability/matched_extratrees_valid_targets"
)

FINAL_MODEL_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/models/"
    "transferability/matched_extratrees_valid_targets"
)

Path(FINAL_DIR).mkdir(parents=True, exist_ok=True)
Path(FINAL_MODEL_DIR).mkdir(parents=True, exist_ok=True)

print("Corrected results directory:", FINAL_DIR)
print("Corrected model directory:", FINAL_MODEL_DIR)

print("Existing result CSVs:", len(list(Path(FINAL_DIR).glob("*.csv"))))
print("Existing model files:", len(list(Path(FINAL_MODEL_DIR).glob("*"))))

Corrected results directory: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_valid_targets
Corrected model directory: /content/drive/MyDrive/AeroSense_AI/models/transferability/matched_extratrees_valid_targets
Existing result CSVs: 0
Existing model files: 0


In [21]:
import inspect

print(inspect.getsource(run_final_extratrees))

def run_final_extratrees(held_out_station, horizon):

    result_file = os.path.join(
        FINAL_DIR,
        f"matched_extratrees_{held_out_station}_{horizon}h.csv"
    )

    model_file = os.path.join(
        FINAL_MODEL_DIR,
        f"matched_extratrees_{held_out_station}_{horizon}h.joblib"
    )

    if os.path.exists(result_file):
        print(f"Already completed: {held_out_station}, {horizon}h")
        return pd.read_csv(result_file)

    print("=" * 65)
    print(f"Training ExtraTrees: {held_out_station} | Horizon: {horizon}h")
    print("=" * 65)

    start_time = time.time()

    X_train, y_train, X_val, y_val = build_pooled_training_data(
        held_out_station, horizon
    )

    test_data = build_matched_sequences(
        held_out_station, horizon
    )

    X_test = test_data["X_test"]
    y_test = test_data["y_test"]

    print("Training samples:", len(y_train))
    print("Validation samples:", len(y_val))
    print("Test samples:", len(y_test))

    model = Extr

In [22]:
import inspect

print(inspect.getsource(build_pooled_training_data))

def build_pooled_training_data(held_out_station, horizon):

    train_X, train_y = [], []
    val_X, val_y = [], []

    for station in ALL_STATIONS:

        if station == held_out_station:
            continue

        data = build_matched_sequences(station, horizon)

        train_X.append(data["X_train"])
        train_y.append(data["y_train"])

        val_X.append(data["X_val"])
        val_y.append(data["y_val"])

    return (
        np.concatenate(train_X),
        np.concatenate(train_y),
        np.concatenate(val_X),
        np.concatenate(val_y)
    )



In [23]:
pilot_result = run_final_extratrees(
    held_out_station="Aotizhongxin",
    horizon=1
)

print("\nPILOT EXPERIMENT COMPLETED")
print(pilot_result.to_string(index=False))

Training ExtraTrees: Aotizhongxin | Horizon: 1h
Training samples: 226706
Validation samples: 37585
Test samples: 5173

RESULT
     model held_out_station  horizon_hours       mae      rmse       r2  training_seconds  test_samples
ExtraTrees     Aotizhongxin              1 10.768094 20.306912 0.952752        478.800172          5173

Saved: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_valid_targets/matched_extratrees_Aotizhongxin_1h.csv
Elapsed time: 7.98 minutes

PILOT EXPERIMENT COMPLETED
     model held_out_station  horizon_hours       mae      rmse       r2  training_seconds  test_samples
ExtraTrees     Aotizhongxin              1 10.768094 20.306912 0.952752        478.800172          5173


In [2]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

FINAL_DIR = (
    "/content/drive/MyDrive/AeroSense_AI/results/"
    "transferability/matched_extratrees_valid_targets"
)

result_files = list(Path(FINAL_DIR).glob("*.csv"))

print("Completed experiments:", len(result_files), "/ 36")
print("Remaining experiments:", 36 - len(result_files))

for file in sorted(result_files):
    print(file.name)

Mounted at /content/drive
Completed experiments: 0 / 36
Remaining experiments: 36


In [3]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

print("Google Drive folders:")
for item in DRIVE.iterdir():
    print(item.name)

print("\nSearching for ExtraTrees results...")

found = list(DRIVE.rglob("matched_extratrees_*.csv"))

print("\nTotal ExtraTrees result files found:", len(found))

for file in found:
    print(file)

Google Drive folders:
III - B  TermII Internal marks.xlsx
Int.MarksTerm2Final.xlsx
Contacts-2024-06-16.vcf
Uma Contacts-2024-06-16.vcf
Certificate.pdf
certificate.pdf
Student details excel  (24-25) iii B.xlsx
Colab Notebooks
Wavs.zip
colab
all_json.zip
all_json
adhi_output_final.zip
janani_output_final.zip
veda_output_final.zip
madhav_output_final.zip
nitheesh_output_final.zip
sasi_output_final.zip
voice
.pdf (1).pdf
.pdf.pdf
Certificate_copy.pdf
Attandance- CBSE Teachers.xlsx
AeroSense_AI

Searching for ExtraTrees results...

Total ExtraTrees result files found: 0


In [4]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")

print("Project folder exists:", PROJECT_DIR.exists())

for item in PROJECT_DIR.iterdir():
    print("[DIR]" if item.is_dir() else "[FILE]", item.name)

print("\nSearching all project CSV files:")

for file in PROJECT_DIR.rglob("*.csv"):
    print(file)

Project folder exists: True
[DIR] data
[DIR] notebooks
[DIR] results
[DIR] models
[DIR] reports

Searching all project CSV files:
/content/drive/MyDrive/AeroSense_AI/results/deep_learning/lstm_gru_comparison.csv
/content/drive/MyDrive/AeroSense_AI/results/deep_learning/extratrees_single_station_comparison.csv
/content/drive/MyDrive/AeroSense_AI/results/deep_learning/all_models_comparison.csv
/content/drive/MyDrive/AeroSense_AI/results/transferability/extratrees_leave_one_station_out.csv
/content/drive/MyDrive/AeroSense_AI/results/transferability/transferability_summary.csv
/content/drive/MyDrive/AeroSense_AI/results/transferability/neural_pilot_Aotizhongxin.csv
/content/drive/MyDrive/AeroSense_AI/results/transferability/neural_pilot_Changping.csv
/content/drive/MyDrive/AeroSense_AI/results/transferability/neural_pilot_Dingling.csv
/content/drive/MyDrive/AeroSense_AI/results/transferability/neural_pilot_Dongsi.csv
/content/drive/MyDrive/AeroSense_AI/results/transferability/neural_pilot_

In [6]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")

notebooks = list(PROJECT_DIR.rglob("*.ipynb"))

print("Total notebooks found:", len(notebooks))

for file in notebooks:
    print(file)

Total notebooks found: 0


In [7]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")

for folder_name in ["data", "models", "reports"]:
    folder = PROJECT_DIR / folder_name
    print(f"\n{'='*20} {folder_name.upper()} {'='*20}")

    files = [f for f in folder.rglob("*") if f.is_file()]
    print("Files found:", len(files))

    for f in files[:40]:
        print(f.relative_to(PROJECT_DIR),
              "|", round(f.stat().st_size / 1024, 2), "KB")


==================== DATA ====================
Files found: 1
data/air_quality_hourly.parquet | 6108.83 KB

==================== MODELS ====================
Files found: 87
models/deep_learning/lstm_1h.pth | 209.84 KB
models/deep_learning/gru_1h.pth | 158.09 KB
models/deep_learning/lstm_6h.pth | 209.84 KB
models/deep_learning/gru_6h.pth | 158.09 KB
models/deep_learning/lstm_24h.pth | 209.98 KB
models/deep_learning/gru_24h.pth | 158.09 KB
models/transferability/lstm_1h_Aotizhongxin.pth | 210.4 KB
models/transferability/gru_1h_Aotizhongxin.pth | 158.64 KB
models/transferability/lstm_6h_Aotizhongxin.pth | 210.4 KB
models/transferability/gru_6h_Aotizhongxin.pth | 158.64 KB
models/transferability/lstm_24h_Aotizhongxin.pth | 210.41 KB
models/transferability/gru_24h_Aotizhongxin.pth | 158.65 KB
models/transferability/lstm_1h_Changping.pth | 211.6 KB
models/transferability/gru_1h_Changping.pth | 159.77 KB
models/transferability/lstm_6h_Changping.pth | 211.6 KB
models/transferability/gru_6h_Ch

In [8]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")
RESULTS_DIR = PROJECT_DIR / "results"

print("Results folder exists:", RESULTS_DIR.exists())

for folder in sorted(RESULTS_DIR.rglob("*")):
    if folder.is_dir():
        files = [f for f in folder.iterdir() if f.is_file()]
        if files:
            print(f"\n{folder.relative_to(PROJECT_DIR)}")
            print("Files:", len(files))
            for f in files[:10]:
                print("  ", f.name)

Results folder exists: True

results/deep_learning
Files: 7
   lstm_gru_comparison.csv
   inference_seconds_comparison.png
   extratrees_single_station_comparison.csv
   all_models_comparison.csv
   mae_comparison.png
   r2_comparison.png
   rmse_comparison.png

results/transferability
Files: 19
   extratrees_leave_one_station_out.csv
   transferability_summary.csv
   station_mae_1h.png
   station_mae_6h.png
   station_mae_24h.png
   neural_pilot_Aotizhongxin.csv
   neural_pilot_Changping.csv
   neural_pilot_Dingling.csv
   neural_pilot_Dongsi.csv
   neural_pilot_Guanyuan.csv

results/transferability/matched_extratrees
Files: 36
   matched_extratrees_Aotizhongxin_1h.csv
   matched_extratrees_Aotizhongxin_6h.csv
   matched_extratrees_Aotizhongxin_24h.csv
   matched_extratrees_Changping_1h.csv
   matched_extratrees_Changping_6h.csv
   matched_extratrees_Changping_24h.csv
   matched_extratrees_Dingling_1h.csv
   matched_extratrees_Dingling_6h.csv
   matched_extratrees_Dingling_24h.csv
   

In [9]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")

files_to_check = [
    PROJECT_DIR / "results/transferability/matched_extratrees_corrected/matched_extratrees_Aotizhongxin_1h.csv",
    PROJECT_DIR / "results/transferability/matched_extratrees_corrected/matched_extratrees_Aotizhongxin_6h.csv",
    PROJECT_DIR / "results/transferability/matched_extratrees_corrected/matched_extratrees_Aotizhongxin_24h.csv",
    PROJECT_DIR / "results/transferability/matched_extratrees_corrected/matched_extratrees_Wanshouxigong_1h.csv",
    PROJECT_DIR / "results/transferability/transferability_summary.csv",
]

for file in files_to_check:
    print("\n" + "=" * 65)
    print(file.name)

    if not file.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(file)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))


matched_extratrees_Aotizhongxin_1h.csv
Shape: (1, 10)
Columns: ['model', 'horizon_hours', 'station', 'train_samples', 'validation_samples', 'test_samples', 'mae', 'rmse', 'r2', 'training_seconds']
     model  horizon_hours      station  train_samples  validation_samples  test_samples       mae      rmse       r2  training_seconds
ExtraTrees              1 Aotizhongxin         226706               37585          5173 10.880356 20.464523 0.952015       1095.064383

matched_extratrees_Aotizhongxin_6h.csv
FILE NOT FOUND

matched_extratrees_Aotizhongxin_24h.csv
FILE NOT FOUND

matched_extratrees_Wanshouxigong_1h.csv
Shape: (1, 8)
Columns: ['model', 'horizon_hours', 'station', 'mae', 'rmse', 'r2', 'training_seconds', 'test_samples']
     model  horizon_hours       station       mae      rmse       r2  training_seconds  test_samples
ExtraTrees              1 Wanshouxigong 12.020012 25.482337 0.938968       1262.632666          5176

transferability_summary.csv
Shape: (3, 7)
Columns: ['model'

In [10]:
from pathlib import Path
import pandas as pd

folder = Path(
    "/content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_corrected"
)

files = sorted(folder.glob("*.csv"))

print("Corrected result files:", len(files))

for file in files:
    print("\n" + "=" * 60)
    print("FILE:", file.name)

    df = pd.read_csv(file)
    print(df.to_string(index=False))

Corrected result files: 4

FILE: matched_extratrees_Aotizhongxin_1h.csv
     model  horizon_hours      station  train_samples  validation_samples  test_samples       mae      rmse       r2  training_seconds
ExtraTrees              1 Aotizhongxin         226706               37585          5173 10.880356 20.464523 0.952015       1095.064383

FILE: matched_extratrees_Wanshouxigong_1h.csv
     model  horizon_hours       station       mae      rmse       r2  training_seconds  test_samples
ExtraTrees              1 Wanshouxigong 12.020012 25.482337 0.938968       1262.632666          5176

FILE: matched_extratrees_Wanshouxigong_24h.csv
     model  horizon_hours       station       mae      rmse       r2  training_seconds  test_samples
ExtraTrees             24 Wanshouxigong 64.753566 97.617207 0.104363       1180.100736          5176

FILE: matched_extratrees_Wanshouxigong_6h.csv
     model  horizon_hours       station       mae      rmse       r2  training_seconds  test_samples
ExtraTrees 

In [11]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")

print("AEROSENSE AI PROJECT STRUCTURE\n")

for item in sorted(PROJECT_DIR.iterdir()):
    if item.is_dir():
        files = [f for f in item.rglob("*") if f.is_file()]
        print(f"[FOLDER] {item.name}/ — {len(files)} files")
    else:
        print(f"[FILE]   {item.name}")

print("\nSOURCE CODE FILES\n")

for pattern in ["*.py", "*.yaml", "*.yml", "*.json", "*.md"]:
    for file in sorted(PROJECT_DIR.rglob(pattern)):
        if any(part.startswith(".") for part in file.parts):
            continue
        print(file.relative_to(PROJECT_DIR))

AEROSENSE AI PROJECT STRUCTURE

[FOLDER] data/ — 1 files
[FOLDER] models/ — 87 files
[FOLDER] notebooks/ — 0 files
[FOLDER] reports/ — 0 files
[FOLDER] results/ — 102 files

SOURCE CODE FILES



In [13]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")
DATA_PATH = PROJECT_DIR / "data/air_quality_hourly.parquet"

FINAL_DIR = PROJECT_DIR / "results/transferability/matched_extratrees_corrected"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_DIR.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Corrected result folder:", FINAL_DIR)
print("Existing corrected experiments:", len(list(FINAL_DIR.glob("*.csv"))))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists: True
Dataset exists: True
Corrected result folder: /content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_corrected
Existing corrected experiments: 4


In [14]:
import pandas as pd
from pathlib import Path

FINAL_DIR = Path(
    "/content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_corrected"
)

results = []

for file in sorted(FINAL_DIR.glob("*.csv")):
    df_result = pd.read_csv(file)
    results.append(df_result)

audit = pd.concat(results, ignore_index=True)

print("Total saved experiments:", len(audit))
print("\nExperiments by horizon:")
print(audit["horizon_hours"].value_counts().sort_index())

print("\nResults:")
print(audit.to_string(index=False))

print("\nDuplicate station-horizon combinations:",
      audit.duplicated(["station", "horizon_hours"]).sum())

Total saved experiments: 4

Experiments by horizon:
horizon_hours
1     2
6     1
24    1
Name: count, dtype: int64

Results:
     model  horizon_hours       station  train_samples  validation_samples  test_samples       mae      rmse       r2  training_seconds
ExtraTrees              1  Aotizhongxin       226706.0             37585.0          5173 10.880356 20.464523 0.952015       1095.064383
ExtraTrees              1 Wanshouxigong            NaN                 NaN          5176 12.020012 25.482337 0.938968       1262.632666
ExtraTrees             24 Wanshouxigong            NaN                 NaN          5176 64.753566 97.617207 0.104363       1180.100736
ExtraTrees              6 Wanshouxigong            NaN                 NaN          5176 37.550527 64.661184 0.607023       1172.322140

Duplicate station-horizon combinations: 0


In [15]:
from pathlib import Path
import pandas as pd

FINAL_DIR = Path(
    "/content/drive/MyDrive/AeroSense_AI/results/transferability/matched_extratrees_corrected"
)

ALL_STATIONS = [
    "Aotizhongxin", "Changping", "Dingling", "Dongsi",
    "Guanyuan", "Gucheng", "Huairou", "Nongzhanguan",
    "Shunyi", "Tiantan", "Wanliu", "Wanshouxigong"
]

HORIZONS = [1, 6, 24]

completed = set()

for file in FINAL_DIR.glob("*.csv"):
    df_result = pd.read_csv(file)
    for _, row in df_result.iterrows():
        completed.add((row["station"], int(row["horizon_hours"])))

remaining = [
    (station, horizon)
    for station in ALL_STATIONS
    for horizon in HORIZONS
    if (station, horizon) not in completed
]

print("Completed:", len(completed))
print("Remaining:", len(remaining))
print("\nRemaining experiments:")

for station, horizon in remaining:
    print(f"{station} | {horizon}h")

Completed: 4
Remaining: 32

Remaining experiments:
Aotizhongxin | 6h
Aotizhongxin | 24h
Changping | 1h
Changping | 6h
Changping | 24h
Dingling | 1h
Dingling | 6h
Dingling | 24h
Dongsi | 1h
Dongsi | 6h
Dongsi | 24h
Guanyuan | 1h
Guanyuan | 6h
Guanyuan | 24h
Gucheng | 1h
Gucheng | 6h
Gucheng | 24h
Huairou | 1h
Huairou | 6h
Huairou | 24h
Nongzhanguan | 1h
Nongzhanguan | 6h
Nongzhanguan | 24h
Shunyi | 1h
Shunyi | 6h
Shunyi | 24h
Tiantan | 1h
Tiantan | 6h
Tiantan | 24h
Wanliu | 1h
Wanliu | 6h
Wanliu | 24h


In [16]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")
DATA_PATH = PROJECT_DIR / "data/air_quality_hourly.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

print("\nStations:", df["station"].nunique())
print("Station names:", sorted(df["station"].unique()))

print("\nTimestamp range:")
print(df["timestamp"].min(), "to", df["timestamp"].max())

Dataset shape: (420768, 14)

Columns:
['timestamp', 'PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'wd', 'WSPM', 'station']

Data types:
timestamp    datetime64[ns]
PM2.5               float64
PM10                float64
SO2                 float64
NO2                 float64
CO                  float64
O3                  float64
TEMP                float64
PRES                float64
DEWP                float64
RAIN                float64
wd                   object
WSPM                float64
station              object
dtype: object

First 5 rows:


,timestamp,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,2013-03-01 00:00:00,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2013-03-01 01:00:00,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,2013-03-01 02:00:00,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,2013-03-01 03:00:00,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,2013-03-01 04:00:00,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin



Stations: 12
Station names: ['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan', 'Wanliu', 'Wanshouxigong']

Timestamp range:
2013-03-01 00:00:00 to 2017-02-28 23:00:00


In [17]:
import torch
import os

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("No GPU detected. Change Colab runtime to T4 GPU or better.")

print("PyTorch version:", torch.__version__)

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
PyTorch version: 2.14.1+cu130


In [18]:
import torch
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

PROJECT_DIR = Path("/content/drive/MyDrive/AeroSense_AI")

TIMESFM_DIR = PROJECT_DIR / "models" / "timesfm_finetuned"
TIMESFM_RESULTS = PROJECT_DIR / "results" / "timesfm"

TIMESFM_DIR.mkdir(parents=True, exist_ok=True)
TIMESFM_RESULTS.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0))
print("Allocated memory:",
      round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
print("Reserved memory:",
      round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")

print("\nCheckpoint directory:", TIMESFM_DIR)
print("Results directory:", TIMESFM_RESULTS)
print("Drive ready:", PROJECT_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
Allocated memory: 0.0 GB
Reserved memory: 0.0 GB

Checkpoint directory: /content/drive/MyDrive/AeroSense_AI/models/timesfm_finetuned
Results directory: /content/drive/MyDrive/AeroSense_AI/results/timesfm
Drive ready: True


In [19]:
%pip install -q "timesfm[torch]>=2.0.0" peft accelerate transformers

In [20]:
import timesfm
import torch
import peft
import transformers

print("TimesFM version:", getattr(timesfm, "__version__", "Version not exposed"))
print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

print("\nTimesFM module location:", timesfm.__file__)
print("GPU available:", torch.cuda.is_available())

TimesFM version: Version not exposed
PyTorch: 2.14.1+cu130
PEFT: 0.21.2
Transformers: 5.22.1

TimesFM module location: /usr/local/lib/python3.13/dist-packages/timesfm/__init__.py
GPU available: True


In [21]:
import torch
import transformers
import peft

print("Checking TimesFM 2.5 model class...")

try:
    from transformers import TimesFm2_5ModelForPrediction
    print("TimesFM 2.5 model class: AVAILABLE")
except Exception as e:
    print("TimesFM model import error:", repr(e))

print("\nChecking PEFT LoRA...")

try:
    from peft import LoraConfig, get_peft_model
    print("LoRA support: AVAILABLE")
except Exception as e:
    print("PEFT import error:", repr(e))

print("\nGPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)

Checking TimesFM 2.5 model class...
TimesFM 2.5 model class: AVAILABLE

Checking PEFT LoRA...
LoRA support: AVAILABLE

GPU: Tesla T4
Transformers: 5.22.1
PEFT: 0.21.2


In [22]:
import pandas as pd
import numpy as np

DATA_PATH = PROJECT_DIR / "data" / "air_quality_hourly.parquet"

df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

station = "Aotizhongxin"

part = (
    df[df["station"] == station]
    .sort_values("timestamp")
    .copy()
)

# Preserve original PM2.5 targets
target = part["PM2.5"].to_numpy(dtype=np.float32)

# Training and validation periods
train_mask = part["timestamp"] < VALIDATION_START
val_mask = (
    (part["timestamp"] >= VALIDATION_START) &
    (part["timestamp"] < TRAIN_END)
)

train_values = target[train_mask.to_numpy()]
val_values = target[val_mask.to_numpy()]

# Remove missing target observations
train_values = train_values[np.isfinite(train_values)]
val_values = val_values[np.isfinite(val_values)]

print("Pilot station:", station)
print("Training observations:", len(train_values))
print("Validation observations:", len(val_values))
print("Training range:", train_values.min(), "to", train_values.max())
print("Validation range:", val_values.min(), "to", val_values.max())
print("Training mean:", round(float(train_values.mean()), 3))
print("Validation mean:", round(float(val_values.mean()), 3))

NameError: name 'VALIDATION_START' is not defined

In [23]:
import pandas as pd
import numpy as np

DATA_PATH = PROJECT_DIR / "data" / "air_quality_hourly.parquet"

VALIDATION_START = pd.Timestamp("2015-07-25 14:00:00")
TRAIN_END = pd.Timestamp("2015-12-18 16:00:00")
TEST_START = pd.Timestamp("2016-07-24 20:00:00")

df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

station = "Aotizhongxin"

part = (
    df[df["station"] == station]
    .sort_values("timestamp")
    .copy()
)

target = part["PM2.5"].to_numpy(dtype=np.float32)

train_mask = part["timestamp"] < VALIDATION_START

val_mask = (
    (part["timestamp"] >= VALIDATION_START) &
    (part["timestamp"] < TRAIN_END)
)

test_mask = part["timestamp"] >= TEST_START

train_values = target[train_mask.to_numpy()]
val_values = target[val_mask.to_numpy()]
test_values = target[test_mask.to_numpy()]

train_values = train_values[np.isfinite(train_values)]
val_values = val_values[np.isfinite(val_values)]
test_values = test_values[np.isfinite(test_values)]

print("Station:", station)
print("Training observations:", len(train_values))
print("Validation observations:", len(val_values))
print("Test observations:", len(test_values))
print("Training mean:", round(float(train_values.mean()), 3))
print("Validation mean:", round(float(val_values.mean()), 3))
print("Test mean:", round(float(test_values.mean()), 3))

Station: Aotizhongxin
Training observations: 20408
Validation observations: 3406
Test observations: 5173
Training mean: 84.205
Validation mean: 80.367
Test mean: 86.943


In [24]:
import numpy as np

CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 24
STRIDE = 24

def create_windows(values, context, horizon, stride):
    X, Y = [], []

    for i in range(0, len(values) - context - horizon + 1, stride):
        x = values[i:i + context]
        y = values[i + context:i + context + horizon]

        if np.isfinite(x).all() and np.isfinite(y).all():
            X.append(x)
            Y.append(y)

    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32)

X_train, y_train = create_windows(
    train_values,
    CONTEXT_LENGTH,
    PREDICTION_LENGTH,
    STRIDE
)

X_val, y_val = create_windows(
    val_values,
    CONTEXT_LENGTH,
    PREDICTION_LENGTH,
    STRIDE
)

print("Training contexts:", X_train.shape)
print("Training targets:", y_train.shape)
print("Validation contexts:", X_val.shape)
print("Validation targets:", y_val.shape)

Training contexts: (829, 512)
Training targets: (829, 24)
Validation contexts: (120, 512)
Validation targets: (120, 24)


In [25]:
from transformers import TimesFm2_5Config

config = TimesFm2_5Config()

print("Model configuration:")
print(config)

print("\nModel class:", TimesFm2_5ModelForPrediction)
print("\nGPU memory available:",
      round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
      "GB")


Model configuration:
TimesFm2_5Config {
  "activation": "swish",
  "attention_bias": false,
  "attention_dropout": 0.0,
  "context_length": 16384,
  "decode_index": 5,
  "force_flip_invariance": true,
  "head_dim": 80,
  "hidden_size": 1280,
  "horizon_length": 128,
  "infer_is_positive": true,
  "initializer_range": 0.02,
  "intermediate_size": 1280,
  "max_position_embeddings": 16384,
  "model_type": "timesfm2_5",
  "num_attention_heads": 16,
  "num_hidden_layers": 20,
  "num_key_value_heads": 16,
  "output_quantile_len": 1024,
  "patch_length": 32,
  "quantiles": [
    0.1,
    0.2,
    0.3,
    0.4,
    0.5,
    0.6,
    0.7,
    0.8,
    0.9
  ],
  "rms_norm_eps": 1e-06,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "transformers_version": "5.22.1",
  "use_bias": false,
  "use_continuous_quantile_head": true
}


Model class: <class 'transformers.models.timesfm2_5.modeling_timesfm2_5.TimesFm2_5ModelForPrediction'>

GPU memory available: 14.56 G

In [26]:
import torch
from transformers import TimesFm2_5ModelForPrediction

MODEL_ID = "google/timesfm-2.5-200m-transformers"

model = TimesFm2_5ModelForPrediction.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
)

model = model.to("cuda")

print("Model loaded successfully.")
print("Device:", next(model.parameters()).device)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "million")
print("GPU allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("GPU reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  925MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model loaded successfully.
Device: cuda:0
Parameters: 231.28928 million
GPU allocated: 0.86 GB
GPU reserved: 0.92 GB


In [27]:
import torch.nn as nn
import inspect

linear_layers = []

for name, module in model.named_modules():
    if isinstance(module, nn.Linear):
        linear_layers.append(name)

print("Total linear layers:", len(linear_layers))

print("\nFirst 40 linear layer names:")
for name in linear_layers[:40]:
    print(name)

print("\nForward method signature:")
print(inspect.signature(model.forward))

print("\nCurrently trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

Total linear layers: 129

First 40 linear layer names:
model.input_ff_layer.input_layer
model.input_ff_layer.output_layer
model.input_ff_layer.residual_layer
model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.0.mlp.fc1
model.layers.0.mlp.fc2
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.1.mlp.fc1
model.layers.1.mlp.fc2
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.2.mlp.fc1
model.layers.2.mlp.fc2
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.3.mlp.fc1
model.layers.3.mlp.fc2
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.4.mlp

In [29]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
import torch
import peft
import transformers

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.14.1+cu130
PEFT: 0.21.2
Transformers: 5.22.1
CUDA available: True
GPU: Tesla T4


In [2]:
import torch
from transformers import TimesFm2_5ModelForPrediction

MODEL_ID = "google/timesfm-2.5-200m-transformers"

model = TimesFm2_5ModelForPrediction.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
)

model = model.to("cuda")

print("TimesFM reloaded successfully")
print("Device:", next(model.parameters()).device)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e6, "million")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

TimesFM reloaded successfully
Device: cuda:0
Parameters: 231.28928 million
GPU memory allocated: 0.86 GB


In [3]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

trainable params: 1,638,400 || all params: 232,927,680 || trainable%: 0.7034
GPU memory allocated: 0.87 GB


In [4]:
# Use two existing training windows
test_context = torch.tensor(
    X_train[:2],
    dtype=torch.float32,
    device="cuda"
)

test_future = torch.tensor(
    y_train[:2],
    dtype=torch.float32,
    device="cuda"
)

model.train()

test_output = model(
    past_values=test_context,
    future_values=test_future,
    forecast_context_len=test_context.shape[1]
)

print("Context shape:", test_context.shape)
print("Future shape:", test_future.shape)
print("Loss:", test_output.loss.item())
print("Mean predictions:", test_output.mean_predictions.shape)
print("Loss finite:", torch.isfinite(test_output.loss).all().item())

NameError: name 'X_train' is not defined

In [5]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/drive/MyDrive/AeroSense_AI/data/air_quality_hourly.parquet"

df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])

station = "Aotizhongxin"
station_df = df[df["station"] == station].sort_values("timestamp").copy()

target = station_df["PM2.5"].to_numpy(dtype=np.float32)
timestamps = station_df["timestamp"]

VALIDATION_START = pd.Timestamp("2015-07-25 14:00:00")
TRAIN_END = pd.Timestamp("2015-12-18 16:00:00")
TEST_START = pd.Timestamp("2016-07-24 20:00:00")

train_mask = timestamps < VALIDATION_START
val_mask = (timestamps >= VALIDATION_START) & (timestamps <= TRAIN_END)
test_mask = timestamps >= TEST_START

train_values = target[train_mask]
val_values = target[val_mask]
test_values = target[test_mask]

CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 24
STRIDE = 24

def create_windows(values, context, horizon, stride):
    X, Y = [], []
    for i in range(0, len(values) - context - horizon + 1, stride):
        x = values[i:i + context]
        y = values[i + context:i + context + horizon]
        if np.isfinite(x).all() and np.isfinite(y).all():
            X.append(x)
            Y.append(y)
    return np.asarray(X, dtype=np.float32), np.asarray(Y, dtype=np.float32)

X_train, y_train = create_windows(
    train_values, CONTEXT_LENGTH, PREDICTION_LENGTH, STRIDE
)

X_val, y_val = create_windows(
    val_values, CONTEXT_LENGTH, PREDICTION_LENGTH, STRIDE
)

print("Station:", station)
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test observations:", len(test_values))

Station: Aotizhongxin
Training: (166, 512) (166, 24)
Validation: (5, 512) (5, 24)
Test observations: 5260


In [6]:
import torch

test_context = torch.tensor(
    X_train[:2], dtype=torch.float32, device="cuda"
)

test_future = torch.tensor(
    y_train[:2], dtype=torch.float32, device="cuda"
)

model.train()

test_output = model(
    past_values=test_context,
    future_values=test_future,
    forecast_context_len=512
)

print("Context shape:", test_context.shape)
print("Future shape:", test_future.shape)
print("Loss:", test_output.loss.item())
print("Prediction shape:", test_output.mean_predictions.shape)
print("Loss finite:", torch.isfinite(test_output.loss).all().item())

TypeError: TimesFm2_5DecoderLayer(
  (self_attn): TimesFm2_5Attention(
    (q_proj): lora.Linear(
      (base_layer): Linear(in_features=1280, out_features=1280, bias=False)
      (lora_dropout): ModuleDict(
        (default): Dropout(p=0.05, inplace=False)
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=1280, out_features=8, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=8, out_features=1280, bias=False)
      )
      (lora_embedding_A): ParameterDict()
      (lora_embedding_B): ParameterDict()
      (lora_magnitude_vector): ModuleDict()
    )
    (k_proj): lora.Linear(
      (base_layer): Linear(in_features=1280, out_features=1280, bias=False)
      (lora_dropout): ModuleDict(
        (default): Dropout(p=0.05, inplace=False)
      )
      (lora_A): ModuleDict(
        (default): Linear(in_features=1280, out_features=8, bias=False)
      )
      (lora_B): ModuleDict(
        (default): Linear(in_features=8, out_features

In [7]:
import torch
from transformers import TimesFm2_5ModelForPrediction
from peft import LoraConfig, get_peft_model

MODEL_ID = "google/timesfm-2.5-200m-transformers"

# Reload clean base model
base_model = TimesFm2_5ModelForPrediction.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32
).to("cuda")

# Official-style LoRA configuration: no task_type
lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

print("Model device:", next(model.parameters()).device)
print("GPU memory:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

trainable params: 1,382,912 || all params: 232,672,192 || trainable%: 0.5944
Model device: cuda:0
GPU memory: 1.74 GB


In [8]:
import torch

test_context = torch.tensor(
    X_train[:2], dtype=torch.float32, device="cuda"
)

test_future = torch.tensor(
    y_train[:2], dtype=torch.float32, device="cuda"
)

model.train()

test_output = model(
    past_values=test_context,
    future_values=test_future,
    forecast_context_len=512
)

print("Loss:", test_output.loss.item())
print("Prediction shape:", test_output.mean_predictions.shape)
print("Loss finite:", torch.isfinite(test_output.loss).all().item())

Loss: 0.2826620936393738
Prediction shape: torch.Size([2, 128])
Loss finite: True


In [9]:
print("Configured horizon:", model.config.horizon_length)
print("Configured context:", model.config.context_length)
print("Prediction output:", test_output.mean_predictions.shape)
print("Target shape:", test_future.shape)
print("Loss:", test_output.loss.item())

print("\nLoRA model type:", type(model).__name__)
print("Base model type:", type(model.get_base_model()).__name__)

Configured horizon: 128
Configured context: 16384
Prediction output: torch.Size([2, 128])
Target shape: torch.Size([2, 24])
Loss: 0.2826620936393738

LoRA model type: PeftModel
Base model type: TimesFm2_5ModelForPrediction
